# Capstone Project — Exploratory Data Analysis

**Problem**: Forecast the next 24 months of U.S. monthly alcohol sales.

This notebook covers the EDA phase:
1. Load and inspect the data
2. DatetimeIndex setup
3. Time plot and summary statistics
4. Seasonal plot and subseries plot
5. Lag plots and autocorrelation
6. STL decomposition
7. Stationarity tests (ADF, KPSS)
8. Differencing
9. ACF/PACF of stationary series
10. Train/test split

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. Load and Inspect

In [ ]:
raw = pd.read_csv(DATA_DIR / "Alcohol_Sales.csv")
print(f"Shape: {raw.shape}")
print(f"Columns: {raw.columns.tolist()}")
print(f"Dtypes:\n{raw.dtypes}")
raw.head()

In [ ]:
raw.tail()

In [ ]:
print(f"Missing values:\n{raw.isnull().sum()}")
print(f"\nBasic stats:")
raw.describe()

---
## 2. DatetimeIndex Setup

In [ ]:
y = pd.Series(
    raw["S4248SM144NCEN"].values,
    index=pd.DatetimeIndex(pd.to_datetime(raw["DATE"]), freq="MS"),
    name="AlcoholSales",
)

print(f"Index type: {type(y.index)}")
print(f"Frequency: {y.index.freq}")
print(f"Date range: {y.index[0].date()} to {y.index[-1].date()}")
print(f"Number of observations: {len(y)}")
y.head()

---
## 3. Time Plot

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
y.plot(ax=ax, color="tab:blue")
ax.set_title("Monthly U.S. Alcohol Sales (Millions $)")
ax.set_xlabel("Date")
ax.set_ylabel("Sales (Millions $)")
plt.tight_layout()
plt.show()

print("Observations:")
print("  - Clear upward trend")
print("  - Strong seasonal pattern with December spikes")
print("  - Seasonal amplitude increases with level (multiplicative seasonality)")
print("  - Possible dip around 2008-2009 (financial crisis)")

---
## 4. Summary Statistics

In [ ]:
print(f"Mean:   {y.mean():.2f}")
print(f"Std:    {y.std():.2f}")
print(f"Min:    {y.min():.2f} ({y.idxmin().date()})")
print(f"Max:    {y.max():.2f} ({y.idxmax().date()})")
print(f"Median: {y.median():.2f}")

# Annual statistics
annual = y.groupby(y.index.year).agg(["mean", "std", "min", "max"])
annual.columns = ["Mean", "Std", "Min", "Max"]
print("\nAnnual summary (first 5 and last 5 years):")
print(pd.concat([annual.head(), annual.tail()]))

---
## 5. Seasonal Plot

Overlay each year to compare seasonal patterns.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

years = y.index.year.unique()
cmap = plt.cm.viridis(np.linspace(0, 1, len(years)))

for year, color in zip(years, cmap):
    subset = y[y.index.year == year]
    ax.plot(subset.index.month, subset.values, color=color, alpha=0.6, linewidth=0.8)

# Highlight first and last years
first_year = y[y.index.year == years[0]]
last_year = y[y.index.year == years[-1]]
ax.plot(first_year.index.month, first_year.values, color="blue", linewidth=2, label=str(years[0]))
ax.plot(last_year.index.month, last_year.values, color="red", linewidth=2, label=str(years[-1]))

ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_title("Seasonal Plot — Alcohol Sales")
ax.set_ylabel("Sales (Millions $)")
ax.legend()
plt.tight_layout()
plt.show()

print("December consistently has the highest sales (holiday season).")
print("The gap between early years (blue) and recent years (red) shows the upward trend.")

---
## 6. Subseries (Box) Plot

Show the distribution of values for each month across all years.

In [ ]:
monthly_data = pd.DataFrame({"month": y.index.month, "sales": y.values})

fig, ax = plt.subplots(figsize=(12, 5))
monthly_data.boxplot(column="sales", by="month", ax=ax)
ax.set_title("Monthly Subseries Plot")
ax.set_xlabel("Month")
ax.set_ylabel("Sales (Millions $)")
plt.suptitle("")  # remove auto-title
plt.tight_layout()
plt.show()

print("December (month 12) has the highest median and widest range.")
print("February tends to have the lowest sales.")

---
## 7. Lag Plots

In [ ]:
from pandas.plotting import lag_plot

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, lag in zip(axes.flat, [1, 3, 6, 12, 24, 36]):
    lag_plot(y, lag=lag, ax=ax, alpha=0.5, s=10)
    ax.set_title(f"Lag {lag}")

plt.suptitle("Lag Plots — Alcohol Sales", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Lag 12 and 24 show strong positive correlation — seasonal pattern.")
print("Lag 1 also shows positive correlation — autoregressive behavior.")

---
## 8. Autocorrelation (ACF/PACF) of Raw Series

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y, lags=48, ax=axes[0], title="ACF — Raw Series")
plot_pacf(y, lags=48, ax=axes[1], title="PACF — Raw Series", method="ywm")
plt.tight_layout()
plt.show()

print("ACF decays slowly — sign of non-stationarity (trend).")
print("Scalloped pattern at lags 12, 24, 36 confirms seasonality.")

---
## 9. STL Decomposition

Seasonal-Trend decomposition using LOESS.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl = STL(y, period=12, robust=True)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

result.observed.plot(ax=axes[0], title="Observed")
result.trend.plot(ax=axes[1], title="Trend")
result.seasonal.plot(ax=axes[2], title="Seasonal")
result.resid.plot(ax=axes[3], title="Residual")

for ax in axes:
    ax.set_ylabel("")

plt.suptitle("STL Decomposition — Alcohol Sales", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Trend: steady upward growth with possible plateau around 2008-2009.")
print("Seasonal: consistent December spikes; pattern shape is stable.")
print("Residual: mostly noise, some larger residuals during financial crisis.")

In [ ]:
# Strength of trend and seasonality
var_resid = result.resid.var()
var_resid_plus_season = (result.resid + result.seasonal).var()
var_resid_plus_trend = (result.resid + result.trend).var()

strength_trend = max(0, 1 - var_resid / var_resid_plus_trend)
strength_season = max(0, 1 - var_resid / var_resid_plus_season)

print(f"Strength of trend:       {strength_trend:.4f}")
print(f"Strength of seasonality: {strength_season:.4f}")
print()
print("Both are close to 1, confirming strong trend and strong seasonality.")

---
## 10. Stationarity Tests

### ADF Test
- $H_0$: The series has a unit root (non-stationary)
- Reject $H_0$ if p-value < 0.05

### KPSS Test
- $H_0$: The series is stationary
- Reject $H_0$ if p-value < 0.05

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def stationarity_tests(series, name=""):
    """Run ADF and KPSS tests and print results."""
    # ADF
    adf_stat, adf_p, adf_lags, adf_nobs, adf_crit, _ = adfuller(series.dropna(), autolag="AIC")
    # KPSS
    kpss_stat, kpss_p, kpss_lags, kpss_crit = kpss(series.dropna(), regression="c", nlags="auto")

    print(f"=== {name} ===")
    print(f"ADF test:  stat={adf_stat:.4f}, p-value={adf_p:.4f}")
    print(f"  -> {'Stationary' if adf_p < 0.05 else 'Non-stationary'} (reject H0 if p < 0.05)")
    print(f"KPSS test: stat={kpss_stat:.4f}, p-value={kpss_p:.4f}")
    print(f"  -> {'Non-stationary' if kpss_p < 0.05 else 'Stationary'} (reject H0 if p < 0.05)")
    print()

stationarity_tests(y, "Raw Series")

---
## 11. Differencing

Apply first differencing to remove the trend, and seasonal differencing
to remove the seasonal component.

In [ ]:
# First difference
y_diff1 = y.diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
y_diff1.plot(ax=axes[0], title="First Difference")
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
plot_acf(y_diff1, lags=48, ax=axes[1], title="ACF — First Difference")
plt.tight_layout()
plt.show()

stationarity_tests(y_diff1, "First Difference")

In [ ]:
# Seasonal difference (lag 12)
y_diff12 = y.diff(12).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
y_diff12.plot(ax=axes[0], title="Seasonal Difference (lag 12)")
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
plot_acf(y_diff12, lags=48, ax=axes[1], title="ACF — Seasonal Difference")
plt.tight_layout()
plt.show()

stationarity_tests(y_diff12, "Seasonal Difference")

In [ ]:
# First + seasonal difference
y_diff1_12 = y.diff().diff(12).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
y_diff1_12.plot(ax=axes[0], title="First + Seasonal Difference")
axes[0].axhline(0, color="red", linestyle="--", alpha=0.5)
plot_acf(y_diff1_12, lags=48, ax=axes[1], title="ACF — First + Seasonal Diff")
plt.tight_layout()
plt.show()

stationarity_tests(y_diff1_12, "First + Seasonal Difference")

---
## 12. ACF/PACF of Stationary Series

Use the differenced series to identify candidate ARIMA orders.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_diff1_12, lags=48, ax=axes[0], title="ACF — First + Seasonal Diff")
plot_pacf(y_diff1_12, lags=48, ax=axes[1], title="PACF — First + Seasonal Diff", method="ywm")
plt.tight_layout()
plt.show()

print("ACF: significant spike at lag 1 and lag 12 suggest MA(1) and SMA(1)")
print("PACF: significant spikes at lags 1, 12 suggest AR(1) and SAR(1)")
print()
print("Candidate SARIMA orders to try:")
print("  SARIMA(1,1,1)(1,1,1,12)")
print("  SARIMA(0,1,1)(0,1,1,12)")
print("  SARIMA(1,1,0)(1,1,0,12)")
print("  Or let auto_arima decide.")

---
## 13. Train/Test Split

Hold out the last 24 months for testing.  This split will be used
consistently in the modelling notebook.

In [ ]:
n_test = 24
y_train = y.iloc[:-n_test]
y_test = y.iloc[-n_test:]

print(f"Train: {len(y_train)} months ({y_train.index[0].date()} to {y_train.index[-1].date()})")
print(f"Test:  {len(y_test)} months ({y_test.index[0].date()} to {y_test.index[-1].date()})")

fig, ax = plt.subplots(figsize=(14, 5))
y_train.plot(ax=ax, label="Train")
y_test.plot(ax=ax, label="Test (holdout)", color="tab:orange")
ax.axvline(y_test.index[0], color="red", linestyle="--", alpha=0.7, label="Split point")
ax.set_title("Train/Test Split — Last 24 Months Held Out")
ax.set_ylabel("Sales (Millions $)")
ax.legend()
plt.tight_layout()
plt.show()

---
## Summary of EDA Findings

| Finding | Detail |
|---------|--------|
| **Trend** | Strong upward trend; strength = ~0.99 |
| **Seasonality** | Strong annual cycle (period=12); December peak, February trough |
| **Seasonality type** | Amplitude grows with level — multiplicative |
| **Stationarity** | Raw series is non-stationary; first + seasonal differencing achieves stationarity |
| **Candidate SARIMA** | SARIMA(p,1,q)(P,1,Q,12) with small p, q, P, Q |
| **Split** | Train: first N-24 months; Test: last 24 months |

These findings guide model selection in the next notebook.

In [ ]:
print("EDA complete.")
print("Next notebook: Capstone modelling — fit, evaluate, and compare all models.")